In [1]:
import numpy as np
import pandas as pd
import rasterio as rio
from rasterio.transform import Affine
from matplotlib import pyplot as plt
import os
from tqdm import tqdm
from pathlib import Path

In [2]:
names = locals()

In [3]:
habitats_lv2_ = rio.open(r'../results/terrestrial-habitat-class-maps/habitat-classifications-lv2-1km.tif')
country_raster_1km_ = rio.open(r"../results/country-rasters/country-raster-eck4-1km.tif")
DEM_ = rio.open(r"D:\wty\Dropbox\code\20250304\results\GMTED2010_EckertIV.tif")

In [4]:
Path(r'../results/area-of-habitat').mkdir(parents=True, exist_ok=True)

In [ ]:
def make_AOH_for_a_taxon(taxon):
    Path(os.path.join('..', 'results', 'area-of-habitat', taxon)).mkdir(parents=True, exist_ok=True)
    habitat_preferences = pd.read_csv(os.path.join('..', 'results', 'iucn-assessments', f'habitat-preferences-{taxon}.csv'))
    range_index = pd.read_csv(os.path.join('..', 'results', 'species-range-rasters', f'range-index-{taxon}.csv'))
    elevation_range = pd.read_csv(os.path.join('..', 'results', 'iucn-assessments', f'elevation-range-{taxon}.csv'))
    invalid_index = []
    # range_index['have_a_AOH'] = np.nan

    
    for i in tqdm(range_index.index):
        a_species = range_index.loc[i, 'name']
        a_path = os.path.join('..', 'results', 'species-range-rasters', taxon, f'{a_species}.tif')
        a_range = rio.open(a_path).read(1) 

        try:
            lower_ele = elevation_range[elevation_range['scientific_name'] == a_species]['lower'].values[0]
        except:
            lower_ele = np.nan 

        try:
            upper_ele = elevation_range[elevation_range['scientific_name'] == a_species]['upper'].values[0]
        except:
            upper_ele = np.nan

        all_codes_ = habitat_preferences[habitat_preferences['scientific_name'] == a_species]['code'].values.tolist()
        all_codes = [int(a.split('_')[0]) * 100 + int(a.split('_')[1]) if '_' in a else int(a) * 100 for a in all_codes_]

        y_1 = range_index.loc[i, 'row_min']
        y_2 = range_index.loc[i, 'row_max']
        x_1 = range_index.loc[i, 'col_min']
        x_2 = range_index.loc[i, 'col_max']
        a_window = ((y_1, y_2+1), (x_1, x_2+1)) 

        local_DEM = DEM_.read(1, window=a_window)
        local_habitats = habitats_lv2_.read(1, window=a_window) 

        habitat_mask = np.isin(local_habitats, all_codes)

        if np.isnan(lower_ele) & ~np.isnan(upper_ele):
            elevation_mask = np.where(local_DEM <= upper_ele, True, False) 
        elif ~np.isnan(lower_ele) & np.isnan(upper_ele):
            elevation_mask = np.where(local_DEM >= lower_ele, True, False) 
        elif np.isnan(lower_ele) & np.isnan(upper_ele):
            elevation_mask = np.where(~np.isnan(local_DEM), True, False)
        else: 
            elevation_mask = np.where((local_DEM >= lower_ele) & (local_DEM <= upper_ele), True, False)

        a_AOH = np.where(habitat_mask & elevation_mask, a_range, 0)

        if np.all(a_AOH == 0):
            invalid_index.append(i)
            range_index.loc[i, 'have_a_AOH'] = False

        else:
            range_index.loc[i, 'have_a_AOH'] = True
            output_path = os.path.join('..', 'results', 'area-of-habitat', taxon, f'{a_species}.tif')
            new_transform = country_raster_1km_.transform * Affine.translation(x_1, y_1)
            with rio.open(
                    output_path,
                    'w',
                    driver='GTiff',
                    height=a_AOH.shape[0],
                    width=a_AOH.shape[1],
                    count=1,
                    dtype=a_AOH.dtype, 
                    crs=country_raster_1km_.crs,
                    transform=new_transform,
                    compress='lzw',
                ) as dst:
                    dst.write(a_AOH, 1)
    

    range_index.to_csv(os.path.join('..', 'results', 'area-of-habitat', f'catalogue-{taxon}.csv'), index=False)

In [11]:
make_AOH_for_a_taxon('amphibians')

100%|██████████| 7640/7640 [05:07<00:00, 24.87it/s]


In [11]:
make_AOH_for_a_taxon('birds')

100%|██████████| 10374/10374 [53:49<00:00,  3.21it/s]  


In [12]:
make_AOH_for_a_taxon('mammals')

100%|██████████| 5750/5750 [26:10<00:00,  3.66it/s]  


In [13]:
make_AOH_for_a_taxon('reptiles')

100%|██████████| 9870/9870 [15:59<00:00, 10.29it/s]  


In [ ]:
catalogue_birds = pd.read_csv('../results/area-of-habitat/catalogue-birds.csv', index_col=0)
red_list_category_birds = pd.read_csv('../results/iucn-assessments/redlist-category-birds.csv', index_col=0)
catalogue_birds['category'] = red_list_category_birds['category']
catalogue_birds.to_csv('../results/area-of-habitat/catalogue-birds.csv')